# S4.1 -- OpenSearch Client + Index Configuration

Interactive verification of Spec S4.1: OpenSearch client, index config, query builder, and factory.

In [1]:
import sys
sys.path.insert(0, "../..")

## 1. Index Configuration

In [2]:
from src.services.opensearch.index_config import ARXIV_PAPERS_CHUNKS_MAPPING, HYBRID_RRF_PIPELINE

# Verify chunk mapping structure
assert "settings" in ARXIV_PAPERS_CHUNKS_MAPPING
assert ARXIV_PAPERS_CHUNKS_MAPPING["settings"]["index.knn"] is True
assert ARXIV_PAPERS_CHUNKS_MAPPING["mappings"]["dynamic"] == "strict"

props = ARXIV_PAPERS_CHUNKS_MAPPING["mappings"]["properties"]
assert props["embedding"]["type"] == "knn_vector"
assert props["embedding"]["dimension"] == 1024
assert props["embedding"]["method"]["engine"] == "nmslib"

print(f"Fields in chunk mapping: {list(props.keys())}")
print(f"Embedding config: {props['embedding']['method']}")
print("\n\u2713 Index configuration verified")

Fields in chunk mapping: ['chunk_id', 'arxiv_id', 'paper_id', 'chunk_index', 'chunk_text', 'chunk_word_count', 'start_char', 'end_char', 'embedding', 'title', 'authors', 'abstract', 'categories', 'published_date', 'section_title', 'parent_chunk_id', 'embedding_model', 'created_at', 'updated_at']
Embedding config: {'name': 'hnsw', 'space_type': 'cosinesimil', 'engine': 'nmslib', 'parameters': {'ef_construction': 512, 'm': 16}}

✓ Index configuration verified


In [3]:
# Verify RRF pipeline
assert HYBRID_RRF_PIPELINE["id"] == "hybrid-rrf-pipeline"
processor = HYBRID_RRF_PIPELINE["phase_results_processors"][0]
assert processor["score-ranker-processor"]["combination"]["technique"] == "rrf"
assert processor["score-ranker-processor"]["combination"]["rank_constant"] == 60

print(f"RRF Pipeline ID: {HYBRID_RRF_PIPELINE['id']}")
print(f"RRF rank_constant: {processor['score-ranker-processor']['combination']['rank_constant']}")
print("\n\u2713 RRF pipeline configuration verified")

RRF Pipeline ID: hybrid-rrf-pipeline
RRF rank_constant: 60

✓ RRF pipeline configuration verified


## 2. Query Builder

In [4]:
from src.services.opensearch.query_builder import QueryBuilder
import json

# BM25 query for chunks
builder = QueryBuilder(query="transformer attention mechanism", size=10, search_chunks=True)
body = builder.build()

assert body["size"] == 10
assert "multi_match" in body["query"]["bool"]["must"][0]
mm = body["query"]["bool"]["must"][0]["multi_match"]
assert mm["query"] == "transformer attention mechanism"
assert "chunk_text^3" in mm["fields"]

print("Query body (BM25 chunk search):")
print(json.dumps(body, indent=2)[:500])
print("\n\u2713 Query builder BM25 chunk mode verified")

Query body (BM25 chunk search):
{
  "query": {
    "bool": {
      "must": [
        {
          "multi_match": {
            "query": "transformer attention mechanism",
            "fields": [
              "chunk_text^3",
              "title^2",
              "abstract^1"
            ],
            "type": "best_fields",
            "operator": "or",
            "fuzziness": "AUTO",
            "prefix_length": 2
          }
        }
      ]
    }
  },
  "size": 10,
  "from": 0,
  "track_total_hits": true,
  "_source": {
 

✓ Query builder BM25 chunk mode verified


In [5]:
# Query with category filter
builder = QueryBuilder(query="BERT", categories=["cs.AI", "cs.CL"], search_chunks=True)
body = builder.build()

assert "filter" in body["query"]["bool"]
assert body["query"]["bool"]["filter"][0] == {"terms": {"categories": ["cs.AI", "cs.CL"]}}

print("Category filter applied correctly")
print("\n\u2713 Query builder with category filter verified")

Category filter applied correctly

✓ Query builder with category filter verified


## 3. OpenSearch Client (Mocked)

In [6]:
from unittest.mock import MagicMock, patch
from src.services.opensearch.client import OpenSearchClient

# Create client with mock
mock_settings = MagicMock()
mock_settings.opensearch.host = "http://localhost:9201"
mock_settings.opensearch.index_name = "arxiv-papers"
mock_settings.opensearch.chunk_index_suffix = "chunks"
mock_settings.opensearch.vector_dimension = 1024
mock_settings.opensearch.rrf_pipeline_name = "hybrid-rrf-pipeline"

with patch("src.services.opensearch.client.OpenSearch") as MockOS:
    mock_os = MagicMock()
    MockOS.return_value = mock_os
    client = OpenSearchClient(host="http://localhost:9201", settings=mock_settings)

    # Test health check
    mock_os.cluster.health.return_value = {"status": "green"}
    assert client.health_check() is True

    # Test BM25 search
    mock_os.search.return_value = {
        "hits": {
            "total": {"value": 1},
            "hits": [{"_id": "c1", "_score": 0.95, "_source": {"chunk_text": "Attention is all you need", "arxiv_id": "1706.03762"}}]
        }
    }
    result = client.search_papers(query="attention", size=5)
    assert result["total"] == 1
    assert result["hits"][0]["arxiv_id"] == "1706.03762"

    print(f"Health: {client.health_check()}")
    print(f"Search result: {result['hits'][0]['chunk_text']}")
    print(f"Index name: {client.index_name}")
    print("\n\u2713 OpenSearch client verified (mocked)")

Health: True
Search result: Attention is all you need
Index name: arxiv-papers-chunks

✓ OpenSearch client verified (mocked)


## 4. Factory Functions

In [7]:
from src.services.opensearch.factory import make_opensearch_client_fresh

with patch("src.services.opensearch.factory.OpenSearchClient") as MockClient:
    MockClient.return_value = MagicMock()
    c1 = make_opensearch_client_fresh(settings=mock_settings)
    c2 = make_opensearch_client_fresh(settings=mock_settings)
    # Fresh factory creates new instances each time
    assert MockClient.call_count == 2

print("\u2713 Factory functions verified")

✓ Factory functions verified


## 5. Module Exports

In [8]:
import src.services.opensearch as opensearch_mod

assert hasattr(opensearch_mod, "OpenSearchClient")
assert hasattr(opensearch_mod, "QueryBuilder")
assert hasattr(opensearch_mod, "ARXIV_PAPERS_CHUNKS_MAPPING")
assert hasattr(opensearch_mod, "HYBRID_RRF_PIPELINE")
assert hasattr(opensearch_mod, "make_opensearch_client")
assert hasattr(opensearch_mod, "make_opensearch_client_fresh")

print(f"Exports: {opensearch_mod.__all__}")
print("\n\u2713 All module exports verified")
print("\n=== S4.1 OpenSearch Client -- ALL CHECKS PASSED ===")

Exports: ['ARXIV_PAPERS_CHUNKS_MAPPING', 'HYBRID_RRF_PIPELINE', 'OpenSearchClient', 'QueryBuilder', 'make_opensearch_client', 'make_opensearch_client_fresh']

✓ All module exports verified

=== S4.1 OpenSearch Client -- ALL CHECKS PASSED ===
